### Read the customer data from the Bronze layer, apply the required cleaning and standardization steps, then save the final cleaned output to the Silver layer as a Delta table.

# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length
from pyspark.sql.window import Window

# Read Bronze table


In [0]:
df = spark.table("workspace.bronze.crm_cust_info")


# Display the raw Bronze data first

In [0]:
# Display the raw Bronze data first to understand the file structure
# and identify the data quality issues that need cleaning in the Silver layer.
df.display()

- ============================================================

Data quality issues found in this file:
 1. Some string columns may contain leading or trailing spaces.
 2. Some customer records have null customer IDs.
 3. Some customer keys do not follow the expected customer key format.
 4. The create date column needs to be converted to a proper date type.
 5. Some customer records are duplicated based on the business key.
 6. Gender values contain many nulls and should be standardized.
 7. Column names need to be renamed to cleaner business-friendly names before saving the final Silver table.

============================================================

# Silver Transformations


# Trimming

In [0]:
# Trimming string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# handle invalid customer record

### Validation checks for invalid customer records


In [0]:
# Validation checks before removing invalid customer records
df.select(
    F.count(F.when(col("cst_id").isNull(), True)).alias("null_customer_id_count"),
    F.count(F.when(~col("cst_key").rlike("^AW[0-9]+$"), True)).alias("invalid_customer_key_count"),
    F.count(F.when(col("cst_create_date").isNull(), True)).alias("null_create_date_count")
).display()

### Removing invalid customer records


In [0]:
# Removing records with null customer IDs or invalid customer keys
df = df.filter(
    col("cst_id").isNotNull() & col("cst_key").rlike("^AW[0-9]+$")
)

# Cleaning Date



In [0]:
# Converting create date to proper date format
df = df.withColumn(
    "cst_create_date",
    F.to_date(col("cst_create_date"), "yyyy-MM-dd")
)

# Standardizing customer columns

In [0]:
# Standardizing customer ID, marital status, and gender values
df = (
    df
    .withColumn("cst_id", col("cst_id").cast("int"))
    .withColumn(
        "cst_marital_status",
        F.when(col("cst_marital_status") == "M", "Married")
         .when(col("cst_marital_status") == "S", "Single")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(col("cst_gndr") == "M", "Male")
         .when(col("cst_gndr") == "F", "Female")
         .otherwise("n/a")
    )
)

# check duplicate record on bussiness keys in this file



In [0]:
#Check for duplicate records based on the business key
# to make sure the customer rows are unique before saving to Silver.
df.groupBy("cst_id", "cst_key") \
  .count() \
  .filter(col("count") > 1) \
  .display()

### Removing duplicate customer records by keeping the latest created record


In [0]:
window_spec = Window.partitionBy("cst_id", "cst_key").orderBy(col("cst_create_date").desc())

df = (
    df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

###  Validation after removing duplicates


In [0]:
# Validation checks after removing duplicate customer records
df.groupBy("cst_id", "cst_key") \
  .count() \
  .filter(col("count") > 1) \
  .count()

#rename coulmns


In [0]:
# Renaming columns to cleaner business names
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#quick validation for df



In [0]:
# Display the cleaned dataframe for a quick validation check
df.display()



#writing silver table

In [0]:
# Writing Silver table
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

#check table created or not


In [0]:
%sql
-- Preview the final Silver table
SELECT * FROM workspace.silver.crm_customers LIMIT 10